# Лабораторная работа №16: Оценка и тестирование промптов (DSPy, LangSmith)

**Студент:** [ФИО]  
**Группа:** [Номер группы]

## Цель работы
Изучить систематические подходы к оценке и оптимизации промптов:
- Фреймворк DSPy для программируемой оптимизации промптов
- Платформа LangSmith для мониторинга и тестирования
- Создание датасетов для оценки LLM
- Метрики качества генерации

## Теоретические сведения

### Проблемы промпт-инжиниринга
Традиционный промпт-инжиниринг имеет недостатки:
- Ручная настройка методом проб и ошибок
- Отсутствие воспроизводимости
- Сложность масштабирования
- Зависимость от конкретной модели

### DSPy (Declarative Self-improving Python)
DSPy — фреймворк для программирования LLM:
- Декларативное описание задач через сигнатуры
- Автоматическая оптимизация промптов
- Компиляция в эффективные промпты
- Независимость от конкретной модели

Ключевые концепции:
- **Signatures** — формальное описание ввода/вывода
- **Modules** — переиспользуемые компоненты
- **Optimizers** — алгоритмы улучшения промптов
- **Metrics** — функции оценки качества

### LangSmith
Платформа для разработки LLM-приложений:
- Трассировка вызовов LLM
- Оценка качества ответов
- A/B тестирование промптов
- Мониторинг продакшена
- Управление датасетами

### Метрики оценки LLM
1. **Точность (Accuracy)** — процент правильных ответов
2. **F1 Score** — баланс precision и recall
3. **ROUGE/BLEU** — для задач суммаризации и перевода
4. **Ответственность (Faithfulness)** — соответствие контексту
5. **Релевантность (Relevance)** — полезность ответа
6. **Токсичность** — безопасность контента

## Задание

### Базовый уровень
1. Установите необходимые библиотеки
2. Создайте простой датасет вопрос-ответ для тестирования
3. Реализуйте базовую метрику оценки качества
4. Протестируйте несколько вариантов промптов вручную

### Продвинутый уровень
1. Освойте основы DSPy: Signature, Module, Predict
2. Создайте компилируемый пайплайн для задачи QA
3. Используйте BootstrapFewShot для оптимизации
4. Сравните качество до и после оптимизации

### Индивидуальный уровень
1. Интегрируйте LangSmith для трекинга экспериментов
2. Разработайте кастомные метрики для вашей задачи
3. Создайте автоматический пайплайн оценки промптов
4. Исследуйте влияние размера few-shot примеров на качество

## Ход работы

In [ ]:
# Установка необходимых библиотек
!pip install -q dspy-ai langchain langsmith datasets evaluate scikit-learn

In [ ]:
import dspy
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

print("Библиотеки установлены")

### Часть 1: Создание датасета для оценки

In [ ]:
# Шаг 1: Создание тестового датасета
qa_dataset = [
    {
        "question": "Какая столица Франции?",
        "answer": "Париж",
        "context": "Франция — государство в Западной Европе. Столица — город Париж."
    },
    {
        "question": "Кто написал роман 'Война и мир'?",
        "answer": "Лев Толстой",
        "context": "Роман-эпопея 'Война и мир' был написан Львом Николаевичем Толстым в 1863-1869 годах."
    },
    {
        "question": "В каком году человек впервые полетел в космос?",
        "answer": "1961",
        "context": "12 апреля 1961 года Юрий Гагарин стал первым человеком в мировой истории, совершившим полёт в космическое пространство."
    },
    {
        "question": "Какой химический элемент обозначается как O?",
        "answer": "Кислород",
        "context": "Кислород (O) — химический элемент 16-й группы второго периода периодической системы."
    },
    {
        "question": "Сколько планет в Солнечной системе?",
        "answer": "8",
        "context": "В Солнечной системе насчитывается 8 планет: Меркурий, Венера, Земля, Марс, Юпитер, Сатурн, Уран и Нептун."
    }
]

print(f"Создан датасет из {len(qa_dataset)} примеров")
for i, example in enumerate(qa_dataset[:2], 1):
    print(f"\n{i}. Вопрос: {example['question']}")
    print(f"   Ответ: {example['answer']}")

### Часть 2: Базовая оценка промптов

In [ ]:
# Шаг 2: Функции для оценки качества
def exact_match_score(predicted: str, actual: str) -> float:
    """
    Точное совпадение ответов (с нормализацией)
    """
    pred_norm = predicted.lower().strip()
    actual_norm = actual.lower().strip()
    return 1.0 if pred_norm == actual_norm else 0.0

def fuzzy_match_score(predicted: str, actual: str) -> float:
    """
    Частичное совпадение (содержит ли ответ ключевые слова)
    """
    pred_norm = predicted.lower().strip()
    actual_norm = actual.lower().strip()
    
    # Проверка полного вхождения
    if actual_norm in pred_norm or pred_norm in actual_norm:
        return 1.0
    
    # Проверка по ключевым словам
    actual_words = set(actual_norm.split())
    pred_words = set(pred_norm.split())
    
    if len(actual_words) == 0:
        return 0.0
    
    overlap = len(actual_words & pred_words)
    return overlap / len(actual_words)

def evaluate_prompts(prompts: List[str], dataset: List[Dict]) -> Dict:
    """
    Оценка нескольких промптов на датасете
    """
    results = {}
    
    for prompt_template in prompts:
        scores = []
        
        for example in dataset:
            # Имитация ответа LLM (в реальности здесь будет вызов API)
            # Для демонстрации используем простую эвристику
            question = example['question']
            context = example.get('context', '')
            
            # Простая эвристика для демонстрации
            if "столица" in question.lower():
                predicted = "Париж"
            elif "тоlstой" in question.lower() or "война" in question.lower():
                predicted = "Лев Толстой"
            elif "космос" in question.lower():
                predicted = "1961"
            elif "элемент" in question.lower() or "O" in question:
                predicted = "Кислород"
            elif "планет" in question.lower():
                predicted = "8"
            else:
                predicted = "Неизвестно"
            
            score = fuzzy_match_score(predicted, example['answer'])
            scores.append(score)
        
        avg_score = sum(scores) / len(scores) if scores else 0
        results[prompt_template[:50]] = {
            "avg_score": avg_score,
            "scores": scores
        }
    
    return results

# Примеры промптов для тестирования
test_prompts = [
    "Ответь на вопрос: {question}",
    "Используя контекст: {context}, ответь: {question}",
    "Вопрос: {question}\nКонтекст: {context}\nОтвет:"
]

prompt_results = evaluate_prompts(test_prompts, qa_dataset)

print("=== Результаты оценки промптов ===")
for prompt, metrics in prompt_results.items():
    print(f"\nПромпт: {prompt}...")
    print(f"Средняя точность: {metrics['avg_score']:.2%}")

### Часть 3: Введение в DSPy

In [ ]:
# Шаг 3: Настройка DSPy с языковой моделью
# Используем заглушку для демонстрации (в реальности подключите LM)

# Пример подключения реальной модели:
# lm = dspy.LM('openai/gpt-3.5-turbo', api_key='your-key')
# или
# lm = dspy.HFClientTGI(model="meta-llama/Llama-2-7b-chat-hf", port=...)

# Для демонстрации создадим mock-модель
class MockLM:
    def __call__(self, prompt, **kwargs):
        # Простая эвристика для демонстрации
        if "столица" in prompt.lower():
            return [dspy.Prediction(answer="Париж")]
        elif "тоlstой" in prompt.lower() or "война" in prompt.lower():
            return [dspy.Prediction(answer="Лев Толстой")]
        elif "космос" in prompt.lower():
            return [dspy.Prediction(answer="1961")]
        else:
            return [dspy.Prediction(answer="Неизвестно")]

lm = MockLM()
print("Модель настроена (mock для демонстрации)")

In [ ]:
# Шаг 4: Определение Signature для задачи
class QuestionAnswerSignature(dspy.Signature):
    """
    Сигнатура для задачи вопрос-ответ с контекстом
    """
    context = dspy.InputField(desc="Контекст для поиска ответа")
    question = dspy.InputField(desc="Вопрос, на который нужно ответить")
    answer = dspy.OutputField(desc="Краткий и точный ответ на вопрос")

print("Сигнатура определена:")
print(f"  Inputs: {list(QuestionAnswerSignature.input_fields.keys())}")
print(f"  Outputs: {list(QuestionAnswerSignature.output_fields.keys())}")

In [ ]:
# Шаг 5: Создание модуля с Predict
class BasicQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prognosticator = dspy.Predict(QuestionAnswerSignature)
    
    def forward(self, context, question):
        prediction = self.prognosticator(context=context, question=question)
        return prediction

# Инициализация модуля
qa_module = BasicQA()

# Тестирование
test_example = qa_dataset[0]
result = qa_module(
    context=test_example['context'],
    question=test_example['question']
)

print(f"\nТест прогноза:")
print(f"  Вопрос: {test_example['question']}")
print(f"  Предсказанный ответ: {result.answer}")
print(f"  Правильный ответ: {test_example['answer']}")

### Часть 4: Оптимизация промптов с DSPy

In [ ]:
# Шаг 6: Определение метрики для оптимизации
def validate_answer(example, pred, trace=None):
    """
    Метрика для оценки качества ответа
    """
    predicted_answer = pred.answer
    actual_answer = example.answer
    
    # Exact match с нормализацией
    score = fuzzy_match_score(predicted_answer, actual_answer)
    return score

print("Метрика валидации определена")

In [ ]:
# Шаг 7: Подготовка данных для обучения
# Конвертация датасета в формат DSPy
train_examples = [
    dspy.Example(
        context=ex['context'],
        question=ex['question'],
        answer=ex['answer']
    ).with_inputs('context', 'question')
    for ex in qa_dataset
]

print(f"Подготовлено {len(train_examples)} примеров для обучения")

In [ ]:
# Шаг 8: Оптимизация с BootstrapFewShot
# Этот оптимизатор автоматически подбирает few-shot примеры

from dspy.teleprompt import BootstrapFewShot

# Настройка оптимизатора
teleprompter = BootstrapFewShot(
    metric=validate_answer,
    max_bootstrapped_demos=2,  # Максимум примеров для few-shot
    max_labeled_demos=2
)

# Компиляция оптимизированного модуля
print("Начало оптимизации...")
optimized_qa = teleprompter.compile(qa_module, trainset=train_examples)
print("Оптимизация завершена!")

# Проверка оптимизированного модуля
test_result = optimized_qa(
    context=test_example['context'],
    question=test_example['question']
)

print(f"\nРезультат после оптимизации:")
print(f"  Предсказанный ответ: {test_result.answer}")

In [ ]:
# Шаг 9: Сравнение базовой и оптимизированной версий
def evaluate_module(module, dataset, module_name):
    scores = []
    
    for example in dataset:
        pred = module(context=example['context'], question=example['question'])
        score = validate_answer(
            dspy.Example(answer=example['answer']),
            pred
        )
        scores.append(score)
    
    avg_score = sum(scores) / len(scores)
    print(f"\n{module_name}:")
    print(f"  Средняя точность: {avg_score:.2%}")
    print(f"  Детальные scores: {[f'{s:.1f}' for s in scores]}")
    
    return avg_score

print("=== Сравнение моделей ===")
baseline_score = evaluate_module(qa_module, qa_dataset, "Базовая модель")
optimized_score = evaluate_module(optimized_qa, qa_dataset, "Оптимизированная модель")

improvement = ((optimized_score - baseline_score) / baseline_score * 100) if baseline_score > 0 else 0
print(f"\nУлучшение: {improvement:+.1f}%")

### Часть 5: Интеграция с LangSmith (демонстрация)

In [ ]:
# Шаг 10: Настройка LangSmith (требует API ключ)
print("""
Для интеграции с LangSmith:

1. Получите API ключ на https://smith.langchain.com
2. Установите переменные окружения:
   export LANGCHAIN_API_KEY="your-key"
   export LANGCHAIN_TRACING_V2="true"
   export LANGCHAIN_PROJECT="my-project"

3. Пример кода для логирования:

from langsmith import Client, traceable

client = Client()

@traceable
def answer_question(question, context):
    # Ваш код для генерации ответа
    return response

# Создание датасета в LangSmith
dataset = client.create_dataset(
    dataset_name="qa-test",
    description="Тестовый датасет для QA"
)

# Добавление примеров
for example in qa_dataset:
    client.create_example(
        inputs={"question": example['question'], "context": example['context']},
        outputs={"answer": example['answer']},
        dataset_id=dataset.id
    )

Примечание: Для работы требуется действительный API ключ.
""")

In [ ]:
# Шаг 11: Создание отчета об оценке
def create_evaluation_report(results: Dict) -> str:
    report = "# Отчет об оценке промптов\n\n"
    report += "## Summary\n\n"
    
    for config, metrics in results.items():
        report += f"### Конфигурация: {config}\n"
        report += f"- Средняя точность: {metrics['avg_score']:.2%}\n"
        report += f"- Количество примеров: {len(metrics['scores'])}\n\n"
    
    report += "## Рекомендации\n\n"
    report += "1. Используйте контекст для повышения точности\n"
    report += "2. Добавляйте few-shot примеры для сложных задач\n"
    report += "3. Тестируйте на репрезентативном датасете\n"
    report += "4. Мониторьте качество в продакшене\n"
    
    return report

report_data = {
    "Baseline": {"avg_score": baseline_score, "scores": []},
    "Optimized": {"avg_score": optimized_score, "scores": []}
}

report = create_evaluation_report(report_data)
print(report)

## Контрольные вопросы

1. Какие проблемы решает фреймворк DSPy по сравнению с ручным промпт-инжинирингом?
2. Что такое Signature в DSPy и зачем она нужна?
3. Как работает оптимизатор BootstrapFewShot?
4. Какие метрики лучше всего подходят для оценки качества ответов LLM?
5. Как LangSmith помогает в разработке и мониторинге LLM-приложений?

## Выводы

В данной лабораторной работе были изучены современные подходы к оценке и оптимизации промптов:
- Создан датасет для систематического тестирования
- Реализованы метрики оценки качества ответов
- Освоены основы фреймворка DSPy (Signatures, Modules, Optimizers)
- Применена автоматическая оптимизация промптов через BootstrapFewShot
- Изучены возможности платформы LangSmith для трекинга экспериментов

Программируемый подход к промптам (DSPy) позволяет автоматизировать процесс оптимизации, повысить воспроизводимость результатов и создавать более надежные LLM-приложения.